# Walmart Weekly Store-Department Forecasting

This notebook trains global sequence models on the Walmart weekly store-sales dataset. It uses store-department series, a chronological split, a seasonal naive baseline, LSTM/Transformer models, and saved evaluation artifacts. Defaults are CPU-friendly and can be expanded for heavier runs.


In [ ]:
import math
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd

torch = None
nn = None
DataLoader = None
TensorDataset = None
device = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def ensure_torch():
    """Import PyTorch only when the neural-model cells are actually run."""
    global torch, nn, DataLoader, TensorDataset, device
    if torch is None:
        try:
            import torch as _torch
            import torch.nn as _nn
            from torch.utils.data import DataLoader as _DataLoader, TensorDataset as _TensorDataset
        except ImportError as exc:
            raise ImportError(
                "PyTorch is required for the neural models. Install torch before running training cells."
            ) from exc
        torch = _torch
        nn = _nn
        DataLoader = _DataLoader
        TensorDataset = _TensorDataset
        torch.manual_seed(SEED)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {device}")
    return torch, nn, DataLoader, TensorDataset, device


In [ ]:
# -------------------------------------------------------------------------
# Config: defaults are intentionally small and CPU-friendly.
# -------------------------------------------------------------------------
WEEKLY_DATA_DIR = Path('data/weekly_store_sales')
WEEKLY_TRAIN_PATH = WEEKLY_DATA_DIR / 'train.csv'
OUTPUT_DIR = Path('outputs/weekly')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FAST_RUN = True
MAX_SERIES = 1500 if FAST_RUN else None
SEQ_LEN = 30
HORIZON = 1
TRAIN_FRAC = 0.8
BATCH_SIZE = 256
EPOCHS = 5 if FAST_RUN else 30
LEARNING_RATE = 1e-3
MIN_SERIES_STD = 1.0
SEASONAL_LAG = 52

print({
    'FAST_RUN': FAST_RUN,
    'MAX_SERIES': MAX_SERIES,
    'SEQ_LEN': SEQ_LEN,
    'EPOCHS': EPOCHS,
    'OUTPUT_DIR': str(OUTPUT_DIR),
})


In [ ]:
def require_file(path: Path, description: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}\n"
            "Place the Walmart weekly train.csv file under data/weekly_store_sales/."
        )
    return path


def mean_absolute_error(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean((y_true - y_pred) ** 2))


def smape(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(np.mean(2.0 * np.abs(y_true - y_pred) / denom) * 100.0)


def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))


def metric_row(model_name, y_true, y_pred):
    return {
        'Model': model_name,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': rmse(y_true, y_pred),
        'sMAPE (%)': smape(y_true, y_pred),
    }


## Data Loading And Cleaning


In [ ]:
require_file(WEEKLY_TRAIN_PATH, 'weekly sales training table')
df_raw = pd.read_csv(WEEKLY_TRAIN_PATH, parse_dates=['Date'])
required_cols = {'Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday'}
missing = required_cols - set(df_raw.columns)
if missing:
    raise ValueError(f'Weekly sales table is missing columns: {sorted(missing)}')

print(df_raw.head())
print(df_raw.shape)


In [ ]:
df = df_raw.copy()
df['id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
df = df.sort_values(['id', 'Date'])

series_order = sorted(df['id'].unique())
if MAX_SERIES is not None:
    rng = np.random.default_rng(SEED)
    series_order = sorted(rng.choice(series_order, size=min(MAX_SERIES, len(series_order)), replace=False))
    df = df[df['id'].isin(series_order)].copy()

sales_pivot = df.pivot_table(index='Date', columns='id', values='Weekly_Sales', aggfunc='sum').sort_index()
holiday_by_date = df.groupby('Date')['IsHoliday'].max().reindex(sales_pivot.index).astype(bool)

# Keep mostly observed series, then interpolate remaining gaps over time.
threshold = int(0.9 * len(sales_pivot))
sales_pivot = sales_pivot.dropna(thresh=threshold, axis=1)
sales_pivot = sales_pivot.interpolate(method='time', axis=0).ffill().bfill()
print(f'Cleaned weekly matrix: {sales_pivot.shape}')


In [ ]:
def weekly_calendar_features(dates, holiday_flags):
    dates = pd.DatetimeIndex(dates)
    week = dates.isocalendar().week.astype(int).to_numpy()
    month = dates.month.to_numpy()
    holiday = np.asarray(holiday_flags, dtype=float)
    return np.stack([
        np.sin(2 * np.pi * week / 52),
        np.cos(2 * np.pi * week / 52),
        np.sin(2 * np.pi * month / 12),
        np.cos(2 * np.pi * month / 12),
        holiday,
    ], axis=1).astype(np.float32)

calendar_features = weekly_calendar_features(sales_pivot.index, holiday_by_date)
N_TIME_FEATS = 1 + calendar_features.shape[1]
print('Calendar feature shape:', calendar_features.shape)


## Window Dataset


In [ ]:
def robust_train_scaler(series, split, min_std=MIN_SERIES_STD):
    train_part = np.asarray(series[:split], dtype=np.float32)
    mu = float(np.nanmean(train_part))
    sd = float(np.nanstd(train_part))
    if not np.isfinite(sd) or sd < min_std:
        sd = float(min_std)
    return mu, sd


def build_global_dataset(series_dict, seq_len, horizon, train_frac):
    X_tr_all, y_tr_all, id_tr_all = [], [], []
    X_te_all, y_te_all, id_te_all = [], [], []
    baseline_te, baseline_pid = [], []
    scalers = {}

    for pid, series in series_dict.items():
        series = np.asarray(series, dtype=np.float32)
        n = len(series)
        split = int(n * train_frac)
        if split <= seq_len + horizon or n - split < horizon:
            continue
        mu, sd = robust_train_scaler(series, split)
        scalers[pid] = (mu, sd)
        scaled = (series - mu) / sd
        full = np.concatenate([scaled[:, None], calendar_features[:n]], axis=1).astype(np.float32)

        for i in range(split - seq_len - horizon + 1):
            X_tr_all.append(full[i:i + seq_len])
            y_tr_all.append(scaled[i + seq_len:i + seq_len + horizon])
            id_tr_all.append(pid)

        for i in range(max(0, split - seq_len), n - seq_len - horizon + 1):
            target_start = i + seq_len
            if target_start < split:
                continue
            X_te_all.append(full[i:i + seq_len])
            y_te_all.append(scaled[target_start:target_start + horizon])
            id_te_all.append(pid)
            lag = SEASONAL_LAG if target_start - SEASONAL_LAG >= 0 else 1
            baseline_te.append(series[target_start - lag:target_start - lag + horizon])
            baseline_pid.append(pid)

    return (
        np.asarray(X_tr_all, dtype=np.float32),
        np.asarray(y_tr_all, dtype=np.float32),
        np.asarray(id_tr_all),
        np.asarray(X_te_all, dtype=np.float32),
        np.asarray(y_te_all, dtype=np.float32),
        np.asarray(id_te_all),
        np.asarray(baseline_te, dtype=np.float32),
        np.asarray(baseline_pid),
        scalers,
    )


ensure_torch()


def make_loader(X, y, batch=BATCH_SIZE, shuffle=True):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch, shuffle=shuffle)


In [ ]:
series_dict = {col: sales_pivot[col].to_numpy(dtype=np.float32) for col in sales_pivot.columns}
(
    X_tr, y_tr, id_tr,
    X_te, y_te, id_te,
    baseline_te, baseline_pid,
    scalers,
) = build_global_dataset(series_dict, SEQ_LEN, HORIZON, TRAIN_FRAC)

if len(X_tr) == 0 or len(X_te) == 0:
    raise ValueError('No train/test windows were created. Reduce SEQ_LEN or check the dataset.')

print(f'Train: X={X_tr.shape}, y={y_tr.shape}')
print(f'Test:  X={X_te.shape}, y={y_te.shape}')
tr_loader = make_loader(X_tr, y_tr, shuffle=True)
te_loader = make_loader(X_te, y_te, shuffle=False)


## Models


In [ ]:
ensure_torch()

class GlobalLSTM(nn.Module):
    def __init__(self, n_time_feats=N_TIME_FEATS, hidden=64, n_layers=2, horizon=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_time_feats,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden, horizon)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1])


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class GlobalTransformer(nn.Module):
    def __init__(self, n_time_feats=N_TIME_FEATS, d_model=64, nhead=4, n_layers=2,
                 dim_ff=128, horizon=1, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_time_feats, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.encoder(x)
        return self.head(x[:, -1])


## Training And Evaluation


In [ ]:
def train(model, tr_loader, te_loader, epochs=EPOCHS, lr=LEARNING_RATE, name='model'):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    crit = nn.MSELoss()
    history = []

    for ep in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            train_loss += loss.item() * xb.size(0)
        train_loss /= len(tr_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in te_loader:
                xb, yb = xb.to(device), yb.to(device)
                val_loss += crit(model(xb), yb).item() * xb.size(0)
        val_loss /= len(te_loader.dataset)
        sched.step()
        history.append({'epoch': ep, 'train_scaled_mse': train_loss, 'val_scaled_mse': val_loss})
        if ep == 1 or ep % 5 == 0 or ep == epochs:
            print(f"[{name}] epoch {ep:3d} | train scaled MSE {train_loss:.4f} | val scaled MSE {val_loss:.4f}")
    return pd.DataFrame(history)


def predict_all(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            preds.append(model(xb.to(device)).cpu().numpy())
            trues.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(trues)


def inverse_scale(values_scaled, pids, scalers):
    values = []
    for value, pid in zip(values_scaled, pids):
        mu, sd = scalers[pid]
        values.append(value * sd + mu)
    return np.asarray(values, dtype=float).ravel()


def evaluate_neural(model, loader, pids, scalers, name):
    preds_s, trues_s = predict_all(model, loader)
    preds = inverse_scale(preds_s, pids, scalers)
    trues = inverse_scale(trues_s, pids, scalers)
    return preds, trues, metric_row(name, trues, preds)


def seasonal_naive_predictions():
    return np.asarray(baseline_te, dtype=float).ravel()


def baseline_metrics():
    trues = inverse_scale(y_te, id_te, scalers)
    preds = seasonal_naive_predictions()
    return preds, trues, metric_row(f'Baseline lag_{SEASONAL_LAG}', trues, preds)


In [ ]:
baseline_pred, baseline_true, baseline_row = baseline_metrics()
print(pd.DataFrame([baseline_row]))


In [ ]:
print('=' * 70)
print('Training GLOBAL LSTM')
lstm = GlobalLSTM(horizon=HORIZON)
lstm_history = train(lstm, tr_loader, te_loader, name='LSTM')

print('=' * 70)
print('Training GLOBAL Transformer')
tfm = GlobalTransformer(horizon=HORIZON)
tfm_history = train(tfm, tr_loader, te_loader, name='Transformer')


In [ ]:
lstm_pred, y_true, lstm_row = evaluate_neural(lstm, te_loader, id_te, scalers, 'LSTM')
tfm_pred, _, tfm_row = evaluate_neural(tfm, te_loader, id_te, scalers, 'Transformer')

model_results = pd.DataFrame([baseline_row, lstm_row, tfm_row]).sort_values('RMSE').reset_index(drop=True)
model_results.to_csv(OUTPUT_DIR / 'weekly_model_results.csv', index=False)
model_results


## Diagnostics And Saved Artifacts


In [ ]:
eval_df = pd.DataFrame({
    'id': id_te,
    'actual': y_true,
    'lstm_pred': lstm_pred,
    'transformer_pred': tfm_pred,
    'baseline_pred': baseline_pred,
})
eval_df[['Store', 'Dept']] = eval_df['id'].str.split('_', expand=True).astype(int)

store_summary = eval_df.groupby('Store').apply(
    lambda g: pd.Series({
        'Transformer_MAE': mean_absolute_error(g['actual'], g['transformer_pred']),
        'Baseline_MAE': mean_absolute_error(g['actual'], g['baseline_pred']),
        'n': len(g),
    })
).reset_index()
department_summary = eval_df.groupby('Dept').apply(
    lambda g: pd.Series({
        'Transformer_MAE': mean_absolute_error(g['actual'], g['transformer_pred']),
        'Baseline_MAE': mean_absolute_error(g['actual'], g['baseline_pred']),
        'n': len(g),
    })
).reset_index()

store_summary.to_csv(OUTPUT_DIR / 'weekly_store_summary.csv', index=False)
department_summary.to_csv(OUTPUT_DIR / 'weekly_department_summary.csv', index=False)
store_summary.head()


In [ ]:
import matplotlib.pyplot as plt


def forecast_vs_actual(example_id=None):
    if example_id is None:
        example_id = eval_df['id'].iloc[0]
    one = eval_df[eval_df['id'] == example_id].reset_index(drop=True)
    if one.empty:
        raise ValueError(f'No evaluation rows for {example_id}')
    plt.figure(figsize=(10, 4))
    plt.plot(one.index, one['actual'], label='Actual', marker='o')
    plt.plot(one.index, one['transformer_pred'], label='Transformer', marker='o')
    plt.plot(one.index, one['baseline_pred'], label=f'Baseline lag_{SEASONAL_LAG}', marker='o')
    plt.title(f'Weekly forecast vs actual: {example_id}')
    plt.xlabel('Evaluation window')
    plt.ylabel('Weekly sales')
    plt.legend()
    plt.tight_layout()
    path = OUTPUT_DIR / 'weekly_forecast_vs_actual.png'
    plt.savefig(path, dpi=150)
    plt.show()
    return path

forecast_plot_path = forecast_vs_actual()
print(forecast_plot_path)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.bar(model_results['Model'], model_results['RMSE'])
plt.xticks(rotation=30, ha='right')
plt.ylabel('RMSE')
plt.title('Weekly model comparison')
plt.tight_layout()
model_comparison_path = OUTPUT_DIR / 'weekly_model_comparison.png'
plt.savefig(model_comparison_path, dpi=150)
plt.show()
print(model_comparison_path)


In [ ]:
with open(OUTPUT_DIR / 'weekly_models.pkl', 'wb') as f:
    pickle.dump({
        'lstm_state_dict': lstm.state_dict(),
        'transformer_state_dict': tfm.state_dict(),
        'scalers': scalers,
        'config': {
            'seq_len': SEQ_LEN,
            'horizon': HORIZON,
            'max_series': MAX_SERIES,
            'fast_run': FAST_RUN,
            'n_time_feats': N_TIME_FEATS,
        },
        'model_results': model_results,
    }, f)

print('Saved:')
print(OUTPUT_DIR / 'weekly_model_results.csv')
print(OUTPUT_DIR / 'weekly_models.pkl')
